# Exercise 04: Logistic Regression and Model Evaluation – Customer Churn

**Dataset:** 7'043 customers of a fictional telecom provider (IBM sample data set based on realistic customer
structures). For each customer we know the contract, the booked services, the charges and whether the customer
**left the company in the last month** (`Churn`). Source:
[Kaggle – Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn).

| variable | meaning |
|---|---|
| `tenure` | number of months the customer has stayed with the company |
| `Contract` | Month-to-month, One year, Two year |
| `InternetService` | DSL, Fiber optic, No |
| `MonthlyCharges`, `TotalCharges` | charges per month / in total (USD) |
| `PaymentMethod`, `PaperlessBilling`, `TechSupport`, `OnlineSecurity`, … | further contract and service details |
| `Churn` | Yes / No – **target** |

**Business question:** Which customers are likely to cancel their contract? The marketing team wants to send a
retention offer to customers at risk. Since an offer costs money and a lost customer costs even more, the
**classification threshold** must be chosen carefully.

**Slides:** Part 04 «Classification models» – logistic regression, evaluation (confusion matrix, metrics, thresholds,
ROC curve and AUC).

**After this exercise you can ...**
- compute probabilities of a logistic regression model by hand,
- fit and interpret a logistic regression in scikit-learn (including categorical features),
- compute and interpret accuracy, precision, recall, F1 and FPR,
- choose a threshold based on costs of false positives and false negatives,
- draw and interpret a ROC curve and the AUC.

## Concept

**Logistic regression.** A linear score is squeezed by the sigmoid function into a probability:

$$P(y=1 \mid x) = \frac{1}{1+e^{-(\beta_0+\beta_1 x_1+\dots+\beta_p x_p)}}$$

$\beta_j > 0$: the probability rises with $x_j$. The model is fitted by maximum likelihood (minimising the log-loss).
(Optional: $e^{\beta_j}$ is the factor by which the *odds* $p/(1-p)$ change per unit of $x_j$.)

**Threshold.** Predict "positive" if $\hat p \ge \tau$ (default $\tau = 0.5$). The threshold should be chosen based on
the costs of false positives vs. false negatives.

**Confusion matrix and metrics.**

| | actual positive | actual negative |
|---|---|---|
| **predicted positive** | TP | FP |
| **predicted negative** | FN | TN |

$$\text{Accuracy}=\frac{TP+TN}{n},\quad \text{Precision}=\frac{TP}{TP+FP},\quad
\text{Recall (TPR)}=\frac{TP}{TP+FN},\quad \text{FPR}=\frac{FP}{FP+TN},\quad
F_1 = 2\cdot\frac{\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}}$$

With imbalanced classes, accuracy alone misleads.

**ROC curve and AUC.** For every threshold $\tau$ plot (FPR, TPR). AUC = area under the curve = probability that a
random positive case gets a higher score than a random negative case (0.5 = random ranking, 1 = perfect).

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, roc_auc_score, classification_report)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Calculations by hand

**a) Probabilities.** A logistic regression with the only feature `tenure` (months with the company) has the
coefficients $\beta_0 = -0.5$ and $\beta_1 = -0.08$. Compute the churn probability for customers with a tenure of
1, 12 and 48 months. Which of them are classified as churners with $\tau = 0.5$ and with $\tau = 0.2$?

**b) Metrics.** A churn model is applied to 1'000 customers, 180 of whom actually churn. It flags 200 customers as
churners, of whom 120 really churn. Fill in the confusion matrix and compute accuracy, precision, recall, FPR and F1.

**c) Baseline.** What accuracy does a "model" reach that simply predicts "no churn" for every customer? What is its
recall?

**d)** Verify your results with Python.

In [ ]:
# a) Probabilities
beta_0, beta_1 = -0.5, -0.08
tenure = np.array([1, 12, 48])
# Your code here

# b) Metrics
TP, FP, FN, TN = ___
# Your code here

**Your answer:** ...

## Exercise 2: Import and prepare the data

a) Import the data. The column `TotalCharges` is stored as text and contains a few blanks. Convert it to a number
(`pd.to_numeric(..., errors='coerce')`) and look at the rows with missing values. What do they have in common? Remove
them.

b) Create the binary target `churn` (1 = Yes, 0 = No). What is the churn rate?

c) Plot the churn rate by `Contract` and by `InternetService`. Interpret.

In [ ]:
# a) Import the data
df = pd.read_csv('./Data/telco_customer_churn.csv', sep=',')

# Your code here

# b) Binary target and churn rate
# Your code here

# c) Churn rate by contract and internet service
# Your code here

**Your answer:** ...

## Exercise 3: Logistic regression with one feature

a) Split the data into training (80 %) and test set (20 %) with `stratify=df['churn']` and `random_state=42`
(split the whole data frame: `train_df, test_df = train_test_split(df, ...)`).

b) Fit a logistic regression `churn ~ tenure` on the training set. Print $\hat\beta_0$ and $\hat\beta_1$ and compare
them with the coefficients of Exercise 1a.

c) Plot the predicted probability against `tenure` (0–72 months) together with the observed churn rate per tenure
(dots). Interpret the S-curve.

In [ ]:
# a) Stratified train/test split
train_df, test_df = ___

# b) Logistic regression with tenure only
log_reg_1 = LogisticRegression()
# Your code here

# c) Plot predicted probability and observed churn rate per tenure
# Your code here

**Your answer:** ...

## Exercise 4: Logistic regression with all features

a) Use all features except `customerID`, `Churn` and `churn`. Create dummy variables for the categorical features with
`pd.get_dummies(..., drop_first=True, dtype=int)` and standardise the features with a `StandardScaler` (fit only on the
training set!).

b) Fit a `LogisticRegression(max_iter=1000)` and plot the 10 largest positive and 10 largest negative coefficients.
Since the features are standardised, the coefficients can be compared with each other. Which characteristics increase
and which decrease the churn probability?

In [ ]:
# a) Features, dummy variables and scaling
features = [c for c in df.columns if c not in ['customerID', 'Churn', 'churn']]
X_train = pd.get_dummies(train_df[features], drop_first=True, dtype=int)
X_test  = pd.get_dummies(test_df[features],  drop_first=True, dtype=int)
y_train, y_test = train_df['churn'], test_df['churn']

scaler = StandardScaler()
X_train_s = ___
X_test_s  = ___

# b) Fit the model and plot the coefficients
log_reg = ___
# Your code here

**Your answer:** ...

## Exercise 5: Evaluation with the default threshold τ = 0.5

a) Predict the test set with `predict()` (= threshold 0.5). Plot the confusion matrix and compute accuracy, precision,
recall, F1 and FPR.

b) Compare the accuracy with the baseline "always no churn". Is the model useful? Which metric is most relevant for the
marketing team?

In [ ]:
# a) Confusion matrix and metrics
y_pred = log_reg.predict(X_test_s)
# Your code here

**Your answer:** ...

## Exercise 6: Choosing the threshold based on costs

The marketing team estimates:
- a **missed churner** (FN) costs **500 USD** (lost margin),
- an **unnecessary retention offer** (FP) costs **100 USD** (discount for a customer who would have stayed anyway).

a) Compute precision, recall, FPR and the total costs $500\cdot FN + 100\cdot FP$ on the test set for the thresholds
$\tau = 0.05, 0.10, \dots, 0.90$ (use `predict_proba`). Show the results as a table and plot the costs against $\tau$.

b) Which threshold minimises the costs? How do recall and precision change compared to $\tau = 0.5$?

*Note: strictly speaking, the threshold should be chosen on validation data (or with CV); here we use the test set to keep
the exercise short.*

In [ ]:
y_prob = log_reg.predict_proba(X_test_s)[:, 1]
cost_fn, cost_fp = 500, 100

rows = []
for tau in np.arange(0.05, 0.91, 0.05):
    y_pred_tau = (y_prob >= tau).astype(int)
    # Your code here

# Your code here

**Your answer:** ...

## Exercise 7: ROC curve and AUC

a) Plot the ROC curve of the model on the test set (`roc_curve`) and compute the AUC (`roc_auc_score`). Mark the points
for $\tau = 0.5$ and for the cost-optimal threshold from Exercise 6. For comparison, add the ROC curve of the simple model
`log_reg_1` (only `tenure`).

b) Check the interpretation of the AUC empirically: draw 100'000 random pairs (one churner, one non-churner) from the test
set and compute the share of pairs in which the churner has the higher predicted probability (ties count ½).

c) Interpret the AUC values.

In [ ]:
# a) ROC curves
# Your code here

# b) AUC as probability of a correct ranking
rng = np.random.default_rng(42)
p_pos = y_prob[y_test.values == 1]
p_neg = y_prob[y_test.values == 0]
# Your code here

**Your answer:** ...

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')